# Show the `lsst.analysis.tools` plots produced by DRP stage2 (recalibrate)

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-06 : stage 2
- purpose : display the plots (dataset types ending with `Plot`) found in the output collection of `DRP.yaml#stage2-recalibrate`, to understand what its steps do
- helpers : `showplots_utils.py` (same folder), shared with the stage 1 notebook
- previous : `nb2026-11-06_ShowPlots_DRPstage1-single-visit.ipynb`

The names of the stage 2 plot dataset types are not known in advance : the notebook discovers them, groups them by kind of plot (`HistPlot`, `FocalPlanePlot`, `WholeSkyPlot`, ...) and lets you browse them.

- the quantum graph for DRP.yaml#stage2-recalibrate: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage2-recalibrate.pdf
- all quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/

## Import

In [ ]:
import os
import re
import logging
import warnings
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

from lsst.daf.butler import Butler

import showplots_utils
importlib.reload(showplots_utils)   # pick up any edit of the helper module without restarting the kernel
from showplots_utils import PlotBrowser, show_diagram

In [ ]:
# Turn the Butler.collections FutureWarning into an error (same convention as the CheckCollections notebooks)
warnings.filterwarnings(
    'error',
    category=FutureWarning,
    message='.*Butler.collections.*',
)

logging.basicConfig(
    level=logging.WARNING,
    format='%(name)s %(levelname)s: %(message)s',
    force=True,
)

butler = Butler('main')

## Configuration

In [ ]:
NB_TAG = 'ShowPlots_DRPstage2'

username = os.getenv('USER')
RUN_NAME = 'runcosmos01_20261103_tract9813patchsel_RunCosmos9patches'
collection_path = f'u/{username}/LSSTCam/{RUN_NAME}/stage2'

# Plot files retrieved from the Butler (a file already present is not downloaded again)
FIGDIR = Path(f'figs_{NB_TAG}')

# Quantum diagrams of the pipeline steps
DIAGRAM_DIR = Path('../../../../notebooks/fig_diagrams')

# Safety limits when displaying many plots
MAX_TYPES = 6     # dataset types displayed per call
MAX_REFS = 12     # plots displayed per dataset type

print('collection :', collection_path)

## Quantum diagrams of the stage2 steps

In [ ]:
DIAGRAM_STEPS = {
    'stage2-recalibrate':           'DRP stage2 : recalibrate (whole stage)',
    'step2a-recalibrate-global':    'DRP step2a/global',
    'step2c-recalibrate-healpix':   'DRP step2c/healpix',
    'step2d-recalibrate-visits':    'DRP step2d/visits',
}

# Steps to display ; a missing file is reported and skipped
SHOW_STEPS = list(DIAGRAM_STEPS)

for step in SHOW_STEPS:
    show_diagram(DIAGRAM_DIR / f'pipeline_drp_pipe_LSSTCam_DRP_{step}.png', title=DIAGRAM_STEPS[step])

## Plots in the stage2 collection

Only the dataset types matching `*Plot` are queried (much faster than listing all the datasets of the collection).

In [ ]:
pb = PlotBrowser(butler, collection_path, raw_dir=FIGDIR)

print('collection            :', collection_path)
print('number of plots       :', len(pb.df))
print('common name prefix    :', repr(pb.prefix))

summary = pb.summary()
with pd.option_context('display.max_colwidth', None, 'display.width', 250, 'display.max_rows', 200):
    display(summary)

### How the plots are split

The `dimensions` column tells at which level each plot is made (per band, per tract, per visit, per detector, whole sky, ...), hence which step of the stage produced it. Number of plots per dataset type and per value of each dimension (only when the number of values is small) :

In [ ]:
for facet in ('band', 'visit', 'detector', 'tract'):
    table = pb.completeness(facet)
    if table.empty:
        continue
    if table.shape[1] <= 12:
        display(Markdown(f'**per {facet}**'))
        display(table)
    else:
        print(f'per {facet} : {table.shape[1]} different values (table not shown)')

### Kinds of plots

In [ ]:
if not summary.empty:
    display(summary.groupby('kind').agg(dataset_types=('n', 'size'), plots=('n', 'sum')))

## Display the plots by kind

One montage per dataset type, one panel per band (or per visit, ...) when the plots are split along that dimension. Reading guide for the usual kinds :

- `HistPlot` : distribution of a metric (repeatability, residuals, ...)
- `FocalPlanePlot` : a quantity binned in focal-plane coordinates (detector- or position-dependent effects)
- `WholeSkyPlot` : a quantity on the sky

Set `KINDS_TO_SHOW` to a list such as `['HistPlot']` to restrict the display ; the limits `MAX_TYPES` / `MAX_REFS` of the configuration cell avoid flooding the notebook.

In [ ]:
KINDS_TO_SHOW = None   # None = all the kinds found, or e.g. ['HistPlot', 'FocalPlanePlot']

kinds = sorted(summary['kind'].unique()) if not summary.empty else []
if KINDS_TO_SHOW is not None:
    kinds = [k for k in kinds if k in KINDS_TO_SHOW]

for kind in kinds:
    display(Markdown(f'### {kind or "other plots"}'))
    pb.show(f'{kind}$' if kind else None, max_types=MAX_TYPES, max_refs=MAX_REFS)

## Browse one dataset type

Pick a dataset type by its row number in the summary table above, and show it as a montage or plot by plot at full resolution. Add `band='i'` (or `visit=...`, `tract=...`) to restrict the selection.

In [ ]:
TYPE_INDEX = 0

if not summary.empty:
    short = summary.index[TYPE_INDEX]
    print(f'[{TYPE_INDEX}] {short}')
    pb.show('^' + re.escape(short) + '$', max_refs=MAX_REFS)
    # one plot at a time, full resolution :
    # pb.show('^' + re.escape(short) + '$', mode='single', max_refs=3)

## Beyond the plots : the metrics behind them

List the metric datasets of the collection, and read one.

In [ ]:
metric_types = pb.sibling_types('metric')
print(f'{len(metric_types)} metric dataset type(s) in the collection :')
for name in metric_types:
    print('  ', name)

In [ ]:
# Pick one of the types listed above
METRIC_TYPE = metric_types[0] if metric_types else None

if METRIC_TYPE is not None:
    metrics = pb.load_first(METRIC_TYPE)
    if hasattr(metrics, 'to_pandas'):
        display(metrics.to_pandas().head())
    else:
        print(str(metrics)[:2000])